# Linear Regression: Capstone

Structure follows [docs/linear_regression_notebook_structure.md](../docs/linear_regression_notebook_structure.md).
Reference implementation: [linear_regression/life_expectancy.ipynb](../linear_regression/life_expectancy.ipynb).

**Rule of thumb:** EDA → clean → engineer → check assumptions → clean again → split and test → train → evaluate with metrics and residual plots → interpret the coefficients.

# Setup

## Environment (local or Colab)

Makes the repo's `utils` package importable.
- **Local:** works from `capstone/` without anything extra (the repo root is added to `sys.path`). `pip install -e .` from the repo root also works.
- **Colab:** set `REPO_URL` below to clone the repo, or upload the `utils/` folder next to this notebook. Upload the dataset too, or point `DATA_FILE` at it.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = ""  # Colab only, e.g. "https://github.com/<user>/devML.git"

if IN_COLAB and REPO_URL and not Path("devML").exists():
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)

# Find the folder that contains utils/ and put it on the import path
for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd() / "devML"):
    if (candidate / "utils").is_dir():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    print("utils/ not found: clone the repo or upload the utils folder.")

print(f"Colab: {IN_COLAB} | cwd: {Path.cwd()}")

## Imports

In [1]:
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

## Reproducibility

`RANDOM_STATE` is passed to sklearn (e.g. `train_test_split`). `np.random.seed` covers any direct `np.random.*` calls.

In [2]:
RANDOM_STATE: int = 42
np.random.seed(RANDOM_STATE)

## Utils

Imported from the repo's `utils` package rather than redefined here.

In [3]:
from utils.base import load_dataset
from utils.eda import (
    check_columns_for_inf,
    check_columns_for_na,
    check_columns_for_nulls,
    check_dataset_size_and_shape,
    dummy_error_to_stop_runs,
    show_columns_with_missing_values,
    show_count_of_columns_missing_values,
)
from utils.tests import (
    test_no_inf_columns,
    test_no_missing_values,
    test_only_numeric_columns,
    test_train_test_split_ratio,
)
from utils.preprocessing import apply_iqr_outlier_removal, remove_rows_with_inf
from utils.plots import (
    plot_actual_vs_predicted,
    plot_boxplots_of_numeric_features,
    plot_predictors_vs_target,
    plot_residual_histogram,
    plot_residuals,
)

# Load Dataset

In [4]:
DATA_FILE: str = "dataset_full.csv"  # path relative to this notebook
TARGET: str = "domain"  # name of the target column

df = load_dataset(DATA_FILE)
df.head()

,source,target,source_length,target_length,readability_score,complexity_level,synthetic_template,domain_context,is_paraphrased,generation_method,quality_flag
0,"Because of data-driven methods, the algorithm ...",": Because of data-driven methods, the algorith...",15,16,22.410000,Medium,T4,Education,True,t5-base,Low
1,"Considering the impact of data preprocessing, ...",: Considering the impact of data preprocessing...,13,14,-40.636923,Low,T2,AI,False,t5-base,Low
2,"Because of data preprocessing, the algorithm i...",": Because of data preprocessing, the algorithm...",16,17,0.245000,Medium,T5,AI,False,t5-base,High
3,"Despite the fact that data preprocessing, rene...","Despite the fact that data preprocessing, rene...",17,17,0.474118,Medium,T1,Climate,False,t5-base,Medium
4,"Because of global warming, the algorithm is ca...",": Because of global warming, the algorithm is ...",15,31,28.050000,Medium,T5,Education,False,t5-base,Low


# STOP - Shift to Classification

In [ ]:
cols_remove = ["synthetic_template", "is_paraphrased", "generation_method","quality_flag"]

# Exploratory Data Analysis (EDA)

## General

In [5]:
check_dataset_size_and_shape(df)
df.info()
df.describe()

Number of rows: 1563
Dataset shape: (1563, 11)
Number of rows (alternative): 1563
<class 'pandas.DataFrame'>
RangeIndex: 1563 entries, 0 to 1562
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   source              1563 non-null   str    
 1   target              1563 non-null   str    
 2   source_length       1563 non-null   int64  
 3   target_length       1563 non-null   int64  
 4   readability_score   1563 non-null   float64
 5   complexity_level    1563 non-null   str    
 6   synthetic_template  1563 non-null   str    
 7   domain_context      1563 non-null   str    
 8   is_paraphrased      1563 non-null   bool   
 9   generation_method   1563 non-null   str    
 10  quality_flag        1563 non-null   str    
dtypes: bool(1), float64(1), int64(2), str(7)
memory usage: 123.8 KB


,source_length,target_length,readability_score
count,1563.000000,1563.000000,1563.000000
mean,14.837492,18.063340,14.503394
std,1.672387,11.066055,21.795486
min,12.000000,0.000000,-50.910000
25%,13.000000,14.000000,-0.150000
50%,15.000000,16.000000,16.107500
75%,16.000000,18.000000,29.467857
max,21.000000,74.000000,68.982500


## Missing Data (NaN, nulls, inf)

In [6]:
missing_cols = show_columns_with_missing_values(df)
columns_with_inf = check_columns_for_inf(df)

Columns with missing values:
Series([], dtype: int64)
Positive inf counts:
source_length        0
target_length        0
readability_score    0
dtype: int64

Negative inf counts:
source_length        0
target_length        0
readability_score    0
dtype: int64
Columns with inf values: []


# Preprocessing

## Dropping

In [ ]:
# TODO: drop columns/rows that can't be used, e.g. df = df.drop(columns=[...])

## Imputing

In [ ]:
# TODO: fill remaining missing values, e.g. df[col] = df[col].fillna(df[col].median())

## Confirm

In [ ]:
test_no_missing_values(df, df.columns.tolist())

# Feature Engineering

Group new features by domain, one sub-header each.

In [ ]:
# TODO: create new features

# 2nd EDA: Assumption Checks

Linear regression assumes linear relationships, little collinearity and numeric inputs. Check these before training.

## Linearity Check

In [ ]:
plot_predictors_vs_target(df, TARGET)

## Multicollinearity

In [ ]:
corr = df.select_dtypes(include="number").corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=90)
ax.set_yticks(range(len(corr.columns)), corr.columns)
fig.colorbar(im)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.show()

# Pairs of predictors above a threshold
CORR_THRESHOLD = 0.8
high_corr = (
    corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    .stack()
    .loc[lambda s: s.abs() > CORR_THRESHOLD]
)
high_corr

## Outliers

In [ ]:
numeric_cols = plot_boxplots_of_numeric_features(df)

## Check Infinites

In [ ]:
columns_with_inf = check_columns_for_inf(df)

# 2nd Preprocessing

## Remove Infinites & Extreme Values

In [ ]:
df = remove_rows_with_inf(df, columns_with_inf)
test_no_inf_columns(check_columns_for_inf(df))

## IQR Outlier Removal

Watch the printed row counts: removing outliers column by column can discard a lot of valid data.

In [ ]:
df = apply_iqr_outlier_removal(df, numeric_cols, exclude={TARGET})
plot_boxplots_of_numeric_features(df, subtitle="Box-Whisker Plots (post-removal)")

## Feature Selection

In [ ]:
# TODO: drop one predictor from each highly correlated pair (see high_corr above)
FEATURES: List[str] = [c for c in df.select_dtypes(include="number").columns if c != TARGET]

## Encoding

In [ ]:
# TODO: encode categorical columns if any are kept, e.g. pd.get_dummies(df, columns=[...], drop_first=True)

## Scaling/Transformation

In [ ]:
# TODO (optional): log-transform skewed features.
# Fit any scaler on the training set only, after the split, to avoid leakage. See Training below.

# Training

## Train/Test Split

In [ ]:
TEST_SIZE: float = 0.25

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
test_only_numeric_columns(X)
test_train_test_split_ratio(df, X_train, X_test, expected_test_size=TEST_SIZE)

## Train

In [ ]:
regressor = LinearRegression()
regressor.fit(X_train, y_train)

# Evaluate

- **MAE**: average error in the target's units; less sensitive to outliers.
- **MSE / RMSE**: penalize large errors; RMSE is back in the target's units.
- **R²**: share of variance explained.
- **MAPE**: average percentage error; undefined if any actual value is 0.

Note which metric matters most for this problem and why.

In [ ]:
def evaluate_model(model, X_test, y_test) -> Dict[str, float]:
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    return {
        "mse": round(float(mse), 3),
        "rmse": round(float(np.sqrt(mse)), 3),
        "mae": round(float(mean_absolute_error(y_test, y_pred)), 3),
        "r2": round(float(r2_score(y_test, y_pred)), 3),
        "mape_pct": round(float(mean_absolute_percentage_error(y_test, y_pred) * 100), 3),
    }

metrics = evaluate_model(regressor, X_test, y_test)
metrics

## Actual vs Predicted

In [ ]:
plot_actual_vs_predicted(regressor, X_test, y_test, target_label=TARGET)

## Residual Scatter Plot

In [ ]:
plot_residuals(regressor, X_test, y_test, target_label=TARGET)

## Residual Histogram

In [ ]:
plot_residual_histogram(regressor, X_test, y_test)

# Feature Influence

Coefficients are only comparable across features if the features are on the same scale.

In [ ]:
coefficients = pd.Series(regressor.coef_, index=FEATURES).sort_values(key=abs, ascending=False)
coefficients

# Next Steps

- TODO